In [1]:
!pip install -q streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 40.0 MB/s eta 0:00:00


In [3]:
import streamlit

In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/2026/comento/week4"
)

print(PROJECT_ROOT.exists())
print(PROJECT_ROOT)

print(list(PROJECT_ROOT.iterdir()))

True
/content/drive/MyDrive/2026/comento/week4
[PosixPath('/content/drive/MyDrive/2026/comento/week4/dataset'), PosixPath('/content/drive/MyDrive/2026/comento/week4/splits'), PosixPath('/content/drive/MyDrive/2026/comento/week4/outputs'), PosixPath('/content/drive/MyDrive/2026/comento/week4/checkpoints'), PosixPath('/content/drive/MyDrive/2026/comento/week4/logs'), PosixPath('/content/drive/MyDrive/2026/comento/week4/data_prepare.ipynb'), PosixPath('/content/drive/MyDrive/2026/comento/week4/utils'), PosixPath('/content/drive/MyDrive/2026/comento/week4/base_utils.ipynb'), PosixPath('/content/drive/MyDrive/2026/comento/week4/01_baseline.ipynb'), PosixPath('/content/drive/MyDrive/2026/comento/week4/02_finetuning.ipynb'), PosixPath('/content/drive/MyDrive/2026/comento/week4/03_evaluation_visualization.ipynb'), PosixPath('/content/drive/MyDrive/2026/comento/week4/04_relative_depth_comparision.ipynb'), PosixPath('/content/drive/MyDrive/2026/comento/week4/streamlit_app')]


In [8]:
# config.py작성
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/2026/comento/week4"
)

STREAMLIT_ROOT = (
    PROJECT_ROOT
    / "streamlit_app"
)
config_path = (
    STREAMLIT_ROOT
    / "config.py"
)

config_code = r'''
from pathlib import Path

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/2026/comento/week4"
)

UTILS_ROOT = (
    PROJECT_ROOT
    / "utils"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "finetuning"
    / "best_model.pt"
)

# ------------------------------------------------------------
# Depth model settings
# ------------------------------------------------------------

MIN_DEPTH = 0.1
MAX_DEPTH = 80.0

HEATMAP_NAME = "inferno"
'''

config_path.write_text(
    config_code,
    encoding="utf-8",
)

print(config_path)
print(config_path.exists())

/content/drive/MyDrive/2026/comento/week4/streamlit_app/config.py
True


In [9]:
from pathlib import Path

checkpoint_candidates = list(
    PROJECT_ROOT.rglob("best_model.pt")
)

print(
    "발견된 best_model.pt:",
    len(checkpoint_candidates),
)

for path in checkpoint_candidates:
    print(path)

발견된 best_model.pt: 1
/content/drive/MyDrive/2026/comento/week4/checkpoints/finetuning/best_model.pt


In [10]:
model_loader_path = (
    STREAMLIT_ROOT
    / "model_loader.py"
)

model_loader_code = r'''
import sys

import streamlit as st
import torch

from config import (
    CHECKPOINT_PATH,
    PROJECT_ROOT,
)


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


import utils.config as project_config
import utils.initialization as init_utils
import utils.inference as inference_utils


@st.cache_resource(
    show_spinner="Fine-tuned Depth Anything V2 모델을 불러오는 중입니다..."
)
def load_finetuned_depth_model():
    """
    Fine-tuned Depth Anything V2 모델과
    image processor를 한 번만 로드합니다.
    """

    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint를 찾을 수 없습니다: "
            f"{CHECKPOINT_PATH}"
        )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model, image_processor = (
        init_utils.load_depth_model(
            model_name=(
                project_config.METRIC_MODEL_NAME
            ),
            device=device,
        )
    )

    checkpoint = (
        inference_utils.load_checkpoint(
            model=model,
            checkpoint_path=CHECKPOINT_PATH,
            device=device,
            optimizer=None,
            strict=True,
        )
    )

    model.eval()

    return {
        "model": model,
        "image_processor": image_processor,
        "device": device,
        "checkpoint": checkpoint,
    }
'''

model_loader_path.write_text(
    model_loader_code,
    encoding="utf-8",
)

print(model_loader_path)

/content/drive/MyDrive/2026/comento/week4/streamlit_app/model_loader.py


In [11]:
inference_path = (
    STREAMLIT_ROOT
    / "inference.py"
)

inference_code = r'''
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from config import (
    MAX_DEPTH,
    MIN_DEPTH,
)


def predict_metric_depth(
    image: Image.Image,
    model,
    image_processor,
    device: torch.device,
) -> np.ndarray:
    """
    PIL RGB 이미지에 대해 metric depth를 예측합니다.

    Returns
    -------
    np.ndarray
        원본 이미지와 동일한 H×W 크기의
        float32 metric depth map
    """

    if image is None:
        raise ValueError(
            "입력 이미지가 없습니다."
        )

    image = image.convert("RGB")

    original_width, original_height = (
        image.size
    )

    inputs = image_processor(
        images=image,
        return_tensors="pt",
    )

    pixel_values = inputs[
        "pixel_values"
    ].to(device)

    with torch.inference_mode():
        outputs = model(
            pixel_values=pixel_values
        )

        predicted_depth = (
            outputs.predicted_depth
        )

        if predicted_depth.ndim == 3:
            predicted_depth = (
                predicted_depth.unsqueeze(1)
            )

        predicted_depth = F.interpolate(
            predicted_depth,
            size=(
                original_height,
                original_width,
            ),
            mode="bicubic",
            align_corners=False,
        )

        predicted_depth = predicted_depth[
            0,
            0,
        ]

        predicted_depth = torch.clamp(
            predicted_depth,
            min=MIN_DEPTH,
            max=MAX_DEPTH,
        )

    return (
        predicted_depth
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )
'''

inference_path.write_text(
    inference_code,
    encoding="utf-8",
)

print(inference_path)

/content/drive/MyDrive/2026/comento/week4/streamlit_app/inference.py


In [12]:
visualization_path = (
    STREAMLIT_ROOT
    / "visualization.py"
)

visualization_code = r'''
import io

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from config import (
    HEATMAP_NAME,
    MAX_DEPTH,
    MIN_DEPTH,
)


def depth_to_heatmap(
    depth_map: np.ndarray,
) -> Image.Image:
    """
    Metric depth map을 컬러바가 포함된
    Heatmap PIL 이미지로 변환합니다.
    """

    if depth_map.ndim != 2:
        raise ValueError(
            "depth_map은 H×W 형태여야 합니다."
        )

    figure, axis = plt.subplots(
        figsize=(10, 5),
    )

    heatmap = axis.imshow(
        depth_map,
        cmap=HEATMAP_NAME,
        vmin=MIN_DEPTH,
        vmax=MAX_DEPTH,
        aspect="auto",
    )

    axis.set_title(
        "Fine-tuned Depth Anything V2"
    )

    axis.axis("off")

    colorbar = figure.colorbar(
        heatmap,
        ax=axis,
        fraction=0.03,
        pad=0.02,
    )

    colorbar.set_label(
        "Predicted depth (m)"
    )

    figure.tight_layout()

    buffer = io.BytesIO()

    figure.savefig(
        buffer,
        format="png",
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(figure)

    buffer.seek(0)

    return Image.open(
        buffer
    ).convert("RGB")
'''

visualization_path.write_text(
    visualization_code,
    encoding="utf-8",
)

print(visualization_path)

/content/drive/MyDrive/2026/comento/week4/streamlit_app/visualization.py


In [13]:
app_path = (
    STREAMLIT_ROOT
    / "app.py"
)

app_code = r'''
import time

import numpy as np
import streamlit as st
from PIL import Image

from inference import predict_metric_depth
from model_loader import (
    load_finetuned_depth_model,
)
from visualization import depth_to_heatmap


st.set_page_config(
    page_title="Object-aware Depth Estimation",
    page_icon="🚗",
    layout="wide",
)


st.title(
    "Object-aware Metric Depth Estimation"
)

st.caption(
    "Fine-tuned Depth Anything V2를 사용해 "
    "업로드한 이미지의 metric depth를 예측합니다."
)


with st.sidebar:
    st.header("모델 정보")

    st.write(
        "Depth model: "
        "Fine-tuned Depth Anything V2"
    )

    st.write(
        "Depth range: 0.1–80 m"
    )

    st.info(
        "현재 버전은 Depth 추론만 지원합니다. "
        "다음 단계에서 YOLOv3 객체 탐지와 "
        "bbox별 거리 추정을 연결합니다."
    )


uploaded_file = st.file_uploader(
    "분석할 이미지를 업로드하세요.",
    type=[
        "jpg",
        "jpeg",
        "png",
        "webp",
    ],
    accept_multiple_files=False,
)


if uploaded_file is None:
    st.info(
        "JPG, PNG 또는 WebP 이미지를 업로드하면 "
        "분석 결과가 표시됩니다."
    )

    st.stop()


try:
    input_image = Image.open(
        uploaded_file
    ).convert("RGB")

except Exception as error:
    st.error(
        f"이미지를 읽을 수 없습니다: {error}"
    )
    st.stop()


st.subheader("입력 이미지")

st.image(
    input_image,
    caption=(
        f"{uploaded_file.name} · "
        f"{input_image.width}×"
        f"{input_image.height}"
    ),
    use_container_width=True,
)


run_inference = st.button(
    "Depth 추론 실행",
    type="primary",
    use_container_width=True,
)


if run_inference:
    try:
        resources = (
            load_finetuned_depth_model()
        )

        start_time = time.perf_counter()

        with st.spinner(
            "Metric depth를 추론하는 중입니다..."
        ):
            depth_map = predict_metric_depth(
                image=input_image,
                model=resources["model"],
                image_processor=(
                    resources[
                        "image_processor"
                    ]
                ),
                device=resources["device"],
            )

            heatmap_image = depth_to_heatmap(
                depth_map
            )

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        st.success(
            f"추론 완료 · "
            f"{elapsed_seconds:.2f}초"
        )

        original_column, depth_column = (
            st.columns(2)
        )

        with original_column:
            st.subheader("Original")

            st.image(
                input_image,
                use_container_width=True,
            )

        with depth_column:
            st.subheader("Depth Heatmap")

            st.image(
                heatmap_image,
                use_container_width=True,
            )

        metric_1, metric_2, metric_3 = (
            st.columns(3)
        )

        finite_depth = depth_map[
            np.isfinite(depth_map)
        ]

        metric_1.metric(
            "Minimum depth",
            f"{finite_depth.min():.2f} m",
        )

        metric_2.metric(
            "Median depth",
            f"{np.median(finite_depth):.2f} m",
        )

        metric_3.metric(
            "Maximum depth",
            f"{finite_depth.max():.2f} m",
        )

        st.caption(
            "현재 값은 Fine-tuned 모델의 metric depth "
            "예측 결과입니다. YOLO 객체별 거리는 다음 "
            "구현 단계에서 bbox 영역의 중앙값으로 계산합니다."
        )

    except Exception as error:
        st.exception(error)
'''

app_path.write_text(
    app_code,
    encoding="utf-8",
)

print(app_path)

/content/drive/MyDrive/2026/comento/week4/streamlit_app/app.py


In [14]:
import py_compile

python_files = [
    STREAMLIT_ROOT / "app.py",
    STREAMLIT_ROOT / "config.py",
    STREAMLIT_ROOT / "inference.py",
    STREAMLIT_ROOT / "model_loader.py",
    STREAMLIT_ROOT / "visualization.py",
]

for path in python_files:
    py_compile.compile(
        str(path),
        doraise=True,
    )

    print("문법 검사 성공:", path.name)

문법 검사 성공: app.py
문법 검사 성공: config.py
문법 검사 성공: inference.py
문법 검사 성공: model_loader.py
문법 검사 성공: visualization.py


In [15]:
import sys

if str(STREAMLIT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(STREAMLIT_ROOT),
    )

import config as app_config

print(
    "Checkpoint:",
    app_config.CHECKPOINT_PATH,
)

print(
    "Exists:",
    app_config.CHECKPOINT_PATH.exists(),
)

Checkpoint: /content/drive/MyDrive/2026/comento/week4/checkpoints/finetuning/best_model.pt
Exists: True
